# 数据库约束设计实习
本次实习的目标是体验如何在数据库中利用各种手段完成数据库约束设计。
## 基本约束设计
我们需要为以下两个表设计约束
- Emp(<u>eno</u>, ename, birthday, level, position, salary, dno)
- Dept(<u>dno</u>, dname, budget, manager)

约束要求：
1. eno和dno是递增序列号形式的主键，长度为4的整型，格式为0001、0002等
2. Emp中的dno为参照Dept的外键，Dept的manager为参照Emp的外键
3. 测试外键定义的三种形式
4. 限定dname为枚举类型（数学学院、计算机学院、智能学院、电子学院、元培学院）
5. 限定position为枚举类型（教师、教务、会计、秘书）
6. 限定level为1到5，默认值为3，salary为2000到200000"

In [1]:
import os
from sqlalchemy import create_engine, text, MetaData, Column, Integer, String, Float, Date, ForeignKey, CheckConstraint
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker, relationship
from datetime import date

In [2]:
# 创建SQLite数据库连接
engine = create_engine('sqlite:///employee_dept.db', echo=True)
metadata = MetaData()
Base = declarative_base()

# 创建会话
Session = sessionmaker(bind=engine)
session = Session()

C:\Users\Dai\AppData\Local\Temp\ipykernel_14120\1065261567.py:4: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()


In [3]:
# 如果已经存在数据库，删除它
if os.path.exists('employee_dept.db'):
    os.remove('employee_dept.db')
    print("删除现有数据库文件")

删除现有数据库文件


In [4]:
# 定义Emp表（放在前面，因为Dept表需要引用它）
class Emp(Base):
    __tablename__ = 'emp'  # 修正了双下划线
    
    eno = Column(String(4), primary_key=True)
    ename = Column(String(50))
    birthday = Column(Date)
    level = Column(Integer, CheckConstraint("level BETWEEN 1 AND 5"), default=3)
    position = Column(String(10), CheckConstraint("position IN ('教师', '教务', '会计', '秘书')"))
    salary = Column(Float, CheckConstraint("salary BETWEEN 2000 AND 200000"))
    dno = Column(String(4), ForeignKey('dept.dno', deferrable=True, initially='DEFERRED'))

    def __repr__(self):
        return f"<Emp(eno='{self.eno}', ename='{self.ename}', level={self.level}, position='{self.position}', salary={self.salary}, dno='{self.dno}')>"
    
    # 注意：我们将在Dept类定义后添加关系

# 定义Dept表
class Dept(Base):
    __tablename__ = 'dept'  # 修正了双下划线
    
    dno = Column(String(4), primary_key=True)
    dname = Column(String(20), CheckConstraint("dname IN ('数学学院', '计算机学院', '智能学院', '电子学院', '元培学院')"))
    budget = Column(Float)
    manager = Column(String(4), ForeignKey('emp.eno', deferrable=True, initially='DEFERRED'))

    def __repr__(self):
        return f"<Dept(dno='{self.dno}', dname='{self.dname}', budget={self.budget}, manager='{self.manager}')>"

# 添加关系引用，解决循环引用问题
Emp.department = relationship("Dept", foreign_keys=[Emp.dno], backref="employees")
Dept.manager_emp = relationship("Emp", foreign_keys=[Dept.manager])


In [5]:
# 创建表
Base.metadata.create_all(engine)

2025-04-19 12:11:39,389 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-19 12:11:39,391 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("emp")
2025-04-19 12:11:39,391 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-04-19 12:11:39,393 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("emp")
2025-04-19 12:11:39,393 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-04-19 12:11:39,395 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("dept")
2025-04-19 12:11:39,395 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-04-19 12:11:39,396 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("dept")
2025-04-19 12:11:39,397 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-04-19 12:11:39,399 INFO sqlalchemy.engine.Engine 
CREATE TABLE emp (
	eno VARCHAR(4) NOT NULL, 
	ename VARCHAR(50), 
	birthday DATE, 
	level INTEGER CHECK (level BETWEEN 1 AND 5), 
	position VARCHAR(10) CHECK (position IN ('教师', '教务', '会计', '秘书')), 
	salary FLOAT CHECK (salary BETWEEN 2000 AND 200000), 
	dno VARCHAR(

In [6]:
# 定义测试函数
def run_test(test_name, test_func):
    """运行测试并打印结果"""
    print(f"\n测试: {test_name}")
    print("-" * 50)
    try:
        test_func()
        print("✓ 测试通过")
    except Exception as e:
        print(f"✗ 测试失败: {e}")
    finally:
        session.rollback()

In [7]:
# 为SQLite创建序列模拟功能
def get_next_id(table_name):
    max_id_query = text(f"SELECT MAX(CAST(SUBSTR({table_name[0]}no, 1, 4) AS INTEGER)) FROM {table_name}")
    result = engine.execute(max_id_query).scalar()
    next_id = 1 if result is None else result + 1
    return f"{next_id:04d}"

In [8]:
# 插入初始数据
try:
    session.begin()
    dept1 = Dept(dno="0001", dname="计算机学院", budget=1000000)
    dept2 = Dept(dno="0002", dname="数学学院", budget=800000)
    dept3 = Dept(dno="0003", dname="智能学院", budget=1200000)
    session.add_all([dept1, dept2, dept3])
    session.flush()

    emp1 = Emp(eno="0001", ename="张三", birthday=date(1980, 1, 15), level=4, position="教师", salary=15000, dno="0001")
    emp2 = Emp(eno="0002", ename="李四", birthday=date(1985, 3, 20), level=3, position="教务", salary=8000, dno="0002")
    emp3 = Emp(eno="0003", ename="王五", birthday=date(1990, 7, 10), level=5, position="教师", salary=20000, dno="0003")
    session.add_all([emp1, emp2, emp3])
    session.flush()

    dept1.manager = "0001"
    dept2.manager = "0002"
    dept3.manager = "0003"

    session.commit()
except Exception as e:
    session.rollback()
    print(f"插入数据时出错: {e}")

2025-04-19 12:11:41,364 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-19 12:11:41,366 INFO sqlalchemy.engine.Engine INSERT INTO dept (dno, dname, budget, manager) VALUES (?, ?, ?, ?)
2025-04-19 12:11:41,367 INFO sqlalchemy.engine.Engine [generated in 0.00080s] [('0001', '计算机学院', 1000000.0, None), ('0002', '数学学院', 800000.0, None), ('0003', '智能学院', 1200000.0, None)]
2025-04-19 12:11:41,370 INFO sqlalchemy.engine.Engine INSERT INTO emp (eno, ename, birthday, level, position, salary, dno) VALUES (?, ?, ?, ?, ?, ?, ?)
2025-04-19 12:11:41,370 INFO sqlalchemy.engine.Engine [generated in 0.00069s] [('0001', '张三', '1980-01-15', 4, '教师', 15000.0, '0001'), ('0002', '李四', '1985-03-20', 3, '教务', 8000.0, '0002'), ('0003', '王五', '1990-07-10', 5, '教师', 20000.0, '0003')]
2025-04-19 12:11:41,374 INFO sqlalchemy.engine.Engine UPDATE dept SET manager=? WHERE dept.dno = ?
2025-04-19 12:11:41,375 INFO sqlalchemy.engine.Engine [generated in 0.00123s] [('0001', '0001'), ('0002', '0002'), ('0003', '00

In [9]:
# 测试默认值
try:
    session.begin()
    emp_default = Emp(eno="0004", ename="吴十", birthday=date(1991, 8, 25), position="秘书", salary=7000, dno="0003")
    session.add(emp_default)
    session.commit()
    emp_result = session.query(Emp).filter_by(ename="吴十").first()
    print(f"默认值测试结果: {emp_result.level}")
except Exception as e:
    session.rollback()
    print(f"测试默认值时出错: {e}")

2025-04-19 12:11:42,175 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-19 12:11:42,177 INFO sqlalchemy.engine.Engine INSERT INTO emp (eno, ename, birthday, level, position, salary, dno) VALUES (?, ?, ?, ?, ?, ?, ?)
2025-04-19 12:11:42,177 INFO sqlalchemy.engine.Engine [generated in 0.00059s] ('0004', '吴十', '1991-08-25', 3, '秘书', 7000.0, '0003')
2025-04-19 12:11:42,192 INFO sqlalchemy.engine.Engine COMMIT
2025-04-19 12:11:42,207 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-19 12:11:42,209 INFO sqlalchemy.engine.Engine SELECT emp.eno AS emp_eno, emp.ename AS emp_ename, emp.birthday AS emp_birthday, emp.level AS emp_level, emp.position AS emp_position, emp.salary AS emp_salary, emp.dno AS emp_dno 
FROM emp 
WHERE emp.ename = ?
 LIMIT ? OFFSET ?
2025-04-19 12:11:42,209 INFO sqlalchemy.engine.Engine [generated in 0.00060s] ('吴十', 1, 0)
默认值测试结果: 3


In [10]:
# 正向测试用例
def test_valid_emp_insert():
    """测试正常员工数据插入"""
    emp = Emp(
        eno="0006", 
        ename="吴六", 
        birthday=date(1990, 5, 15), 
        level=2, 
        position="教师", 
        salary=12000, 
        dno="0001"
    )
    session.add(emp)
    session.commit()

    # 验证插入成功
    result = session.query(Emp).filter_by(eno="0003").first()
    assert result is not None, "员工数据未成功插入"
    print(f"成功插入员工: {result}")
    
run_test("正常员工数据插入", test_valid_emp_insert)


测试: 正常员工数据插入
--------------------------------------------------
2025-04-19 12:11:43,735 INFO sqlalchemy.engine.Engine INSERT INTO emp (eno, ename, birthday, level, position, salary, dno) VALUES (?, ?, ?, ?, ?, ?, ?)
2025-04-19 12:11:43,736 INFO sqlalchemy.engine.Engine [generated in 0.00111s] ('0006', '吴六', '1990-05-15', 2, '教师', 12000.0, '0001')
2025-04-19 12:11:43,741 INFO sqlalchemy.engine.Engine COMMIT
2025-04-19 12:11:43,755 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-19 12:11:43,758 INFO sqlalchemy.engine.Engine SELECT emp.eno AS emp_eno, emp.ename AS emp_ename, emp.birthday AS emp_birthday, emp.level AS emp_level, emp.position AS emp_position, emp.salary AS emp_salary, emp.dno AS emp_dno 
FROM emp 
WHERE emp.eno = ?
 LIMIT ? OFFSET ?
2025-04-19 12:11:43,758 INFO sqlalchemy.engine.Engine [generated in 0.00128s] ('0003', 1, 0)
成功插入员工: <Emp(eno='0003', ename='王五', level=5, position='教师', salary=20000.0, dno='0003')>
✓ 测试通过
2025-04-19 12:11:43,762 INFO sqlalchemy.engine.

In [11]:
def test_valid_dept_insert():
    """测试正常部门数据插入"""
    dept = Dept(
        dno="0005", 
        dname="电子学院", 
        budget=1200000, 
        manager="0004"
    )
    session.add(dept)
    session.commit()
    
    # 验证插入成功
    result = session.query(Dept).filter_by(dno="0003").first()
    assert result is not None, "部门数据未成功插入"
    print(f"成功插入部门: {result}")

run_test("正常部门数据插入", test_valid_dept_insert)


测试: 正常部门数据插入
--------------------------------------------------
2025-04-19 12:11:44,521 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-19 12:11:44,522 INFO sqlalchemy.engine.Engine INSERT INTO dept (dno, dname, budget, manager) VALUES (?, ?, ?, ?)
2025-04-19 12:11:44,522 INFO sqlalchemy.engine.Engine [generated in 0.00065s] ('0005', '电子学院', 1200000.0, '0004')
2025-04-19 12:11:44,525 INFO sqlalchemy.engine.Engine COMMIT
2025-04-19 12:11:44,552 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-19 12:11:44,553 INFO sqlalchemy.engine.Engine SELECT dept.dno AS dept_dno, dept.dname AS dept_dname, dept.budget AS dept_budget, dept.manager AS dept_manager 
FROM dept 
WHERE dept.dno = ?
 LIMIT ? OFFSET ?
2025-04-19 12:11:44,553 INFO sqlalchemy.engine.Engine [generated in 0.00117s] ('0003', 1, 0)
成功插入部门: <Dept(dno='0003', dname='智能学院', budget=1200000.0, manager='0003')>
✓ 测试通过
2025-04-19 12:11:44,555 INFO sqlalchemy.engine.Engine ROLLBACK


In [12]:
def test_default_level():
    """测试员工level默认值为3"""
    emp = Emp(
        eno="0005", 
        ename="赵六", 
        birthday=date(1992, 8, 25), 
        position="秘书", 
        salary=7000, 
        dno="0001"
    )
    session.add(emp)
    session.commit()
    
    # 验证默认值
    result = session.query(Emp).filter_by(eno="0005").first()
    assert result.level == 3, f"默认值错误: 预期为3, 实际为{result.level}"
    print(f"默认值测试成功: {result}")

run_test("员工level默认值测试", test_default_level)


测试: 员工level默认值测试
--------------------------------------------------
2025-04-19 12:11:45,163 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-19 12:11:45,165 INFO sqlalchemy.engine.Engine INSERT INTO emp (eno, ename, birthday, level, position, salary, dno) VALUES (?, ?, ?, ?, ?, ?, ?)
2025-04-19 12:11:45,166 INFO sqlalchemy.engine.Engine [cached since 2.99s ago] ('0005', '赵六', '1992-08-25', 3, '秘书', 7000.0, '0001')
2025-04-19 12:11:45,172 INFO sqlalchemy.engine.Engine COMMIT
2025-04-19 12:11:45,202 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-19 12:11:45,205 INFO sqlalchemy.engine.Engine SELECT emp.eno AS emp_eno, emp.ename AS emp_ename, emp.birthday AS emp_birthday, emp.level AS emp_level, emp.position AS emp_position, emp.salary AS emp_salary, emp.dno AS emp_dno 
FROM emp 
WHERE emp.eno = ?
 LIMIT ? OFFSET ?
2025-04-19 12:11:45,206 INFO sqlalchemy.engine.Engine [cached since 1.448s ago] ('0005', 1, 0)
默认值测试成功: <Emp(eno='0005', ename='赵六', level=3, position='秘书', salar

In [13]:
def test_circular_reference():
    """测试循环引用 - 员工引用部门，部门经理引用员工"""
    # 先创建部门，不设置经理
    dept = Dept(dno="0004", dname="元培学院", budget=900000)
    session.add(dept)
    session.flush()
    
    # 创建员工，引用该部门
    emp = Emp(
        eno="0007", 
        ename="钱七", 
        birthday=date(1988, 9, 12), 
        level=5, 
        position="教师", 
        salary=25000, 
        dno="0004"
    )
    session.add(emp)
    session.flush()
    
    # 将该员工设为部门经理
    dept.manager = "0007"
    session.commit()
    
    # 验证引用关系
    dept_result = session.query(Dept).filter_by(dno="0004").first()
    emp_result = session.query(Emp).filter_by(eno="0007").first()
    
    assert dept_result.manager == "0007", "部门经理未设置成功"
    assert emp_result.dno == "0004", "员工部门未设置成功"
    print(f"循环引用测试成功: 部门{dept_result}, 员工{emp_result}")

run_test("循环引用测试", test_circular_reference)


测试: 循环引用测试
--------------------------------------------------
2025-04-19 12:11:45,812 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-19 12:11:45,812 INFO sqlalchemy.engine.Engine INSERT INTO dept (dno, dname, budget, manager) VALUES (?, ?, ?, ?)
2025-04-19 12:11:45,814 INFO sqlalchemy.engine.Engine [cached since 1.292s ago] ('0004', '元培学院', 900000.0, None)
2025-04-19 12:11:45,820 INFO sqlalchemy.engine.Engine INSERT INTO emp (eno, ename, birthday, level, position, salary, dno) VALUES (?, ?, ?, ?, ?, ?, ?)
2025-04-19 12:11:45,821 INFO sqlalchemy.engine.Engine [cached since 2.086s ago] ('0007', '钱七', '1988-09-12', 5, '教师', 25000.0, '0004')
2025-04-19 12:11:45,823 INFO sqlalchemy.engine.Engine UPDATE dept SET manager=? WHERE dept.dno = ?
2025-04-19 12:11:45,824 INFO sqlalchemy.engine.Engine [generated in 0.00094s] ('0007', '0004')
2025-04-19 12:11:45,825 INFO sqlalchemy.engine.Engine COMMIT
2025-04-19 12:11:45,834 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-19 12:11:45

In [14]:
# 负向测试用例
def test_invalid_level():
    """测试员工level约束（1-5之间）"""
    emp = Emp(
        eno="0009", 
        ename="测试", 
        birthday=date(1990, 5, 15), 
        level=6,  # 超出范围
        position="教师", 
        salary=12000, 
        dno="0001"
    )
    session.add(emp)
    session.commit()  # 应该引发异常
    
run_test("非法员工level值(超过上限)", test_invalid_level)


测试: 非法员工level值(超过上限)
--------------------------------------------------
2025-04-19 12:11:46,653 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-19 12:11:46,653 INFO sqlalchemy.engine.Engine INSERT INTO emp (eno, ename, birthday, level, position, salary, dno) VALUES (?, ?, ?, ?, ?, ?, ?)
2025-04-19 12:11:46,653 INFO sqlalchemy.engine.Engine [cached since 2.918s ago] ('0009', '测试', '1990-05-15', 6, '教师', 12000.0, '0001')
2025-04-19 12:11:46,655 INFO sqlalchemy.engine.Engine ROLLBACK
✗ 测试失败: (sqlite3.IntegrityError) CHECK constraint failed: level BETWEEN 1 AND 5
[SQL: INSERT INTO emp (eno, ename, birthday, level, position, salary, dno) VALUES (?, ?, ?, ?, ?, ?, ?)]
[parameters: ('0009', '测试', '1990-05-15', 6, '教师', 12000.0, '0001')]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


In [15]:
def test_invalid_level_low():
    """测试员工level下限约束"""
    emp = Emp(
        eno="0009", 
        ename="测试", 
        birthday=date(1990, 5, 15), 
        level=0,  # 低于下限
        position="教师", 
        salary=12000, 
        dno="0001"
    )
    session.add(emp)
    session.commit()  # 应该引发异常
run_test("非法员工level值(低于下限)", test_invalid_level)


测试: 非法员工level值(低于下限)
--------------------------------------------------
2025-04-19 12:11:47,467 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-19 12:11:47,468 INFO sqlalchemy.engine.Engine INSERT INTO emp (eno, ename, birthday, level, position, salary, dno) VALUES (?, ?, ?, ?, ?, ?, ?)
2025-04-19 12:11:47,468 INFO sqlalchemy.engine.Engine [cached since 3.733s ago] ('0009', '测试', '1990-05-15', 6, '教师', 12000.0, '0001')
2025-04-19 12:11:47,469 INFO sqlalchemy.engine.Engine ROLLBACK
✗ 测试失败: (sqlite3.IntegrityError) CHECK constraint failed: level BETWEEN 1 AND 5
[SQL: INSERT INTO emp (eno, ename, birthday, level, position, salary, dno) VALUES (?, ?, ?, ?, ?, ?, ?)]
[parameters: ('0009', '测试', '1990-05-15', 6, '教师', 12000.0, '0001')]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


In [16]:
def test_invalid_position():
    """测试职位枚举约束"""
    emp = Emp(
        eno="0009", 
        ename="测试", 
        birthday=date(1990, 5, 15), 
        level=3, 
        position="主任",  # 不在枚举列表中
        salary=12000, 
        dno="0001"
    )
    session.add(emp)
    session.commit()  # 应该引发异常
run_test("非法职位值", test_invalid_position)


测试: 非法职位值
--------------------------------------------------
2025-04-19 12:11:48,525 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-19 12:11:48,526 INFO sqlalchemy.engine.Engine INSERT INTO emp (eno, ename, birthday, level, position, salary, dno) VALUES (?, ?, ?, ?, ?, ?, ?)
2025-04-19 12:11:48,527 INFO sqlalchemy.engine.Engine [cached since 4.791s ago] ('0009', '测试', '1990-05-15', 3, '主任', 12000.0, '0001')
2025-04-19 12:11:48,527 INFO sqlalchemy.engine.Engine ROLLBACK
✗ 测试失败: (sqlite3.IntegrityError) CHECK constraint failed: position IN ('教师', '教务', '会计', '秘书')
[SQL: INSERT INTO emp (eno, ename, birthday, level, position, salary, dno) VALUES (?, ?, ?, ?, ?, ?, ?)]
[parameters: ('0009', '测试', '1990-05-15', 3, '主任', 12000.0, '0001')]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


In [17]:
def test_invalid_salary_low():
    """测试薪资下限约束"""
    emp = Emp(
        eno="0009", 
        ename="测试", 
        birthday=date(1990, 5, 15), 
        level=3, 
        position="教师", 
        salary=1000,  # 低于最低薪资
        dno="0001"
    )
    session.add(emp)
    session.commit()  # 应该引发异常
run_test("非法薪资值(低于下限)", test_invalid_salary_low)


测试: 非法薪资值(低于下限)
--------------------------------------------------
2025-04-19 12:11:50,138 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-19 12:11:50,139 INFO sqlalchemy.engine.Engine INSERT INTO emp (eno, ename, birthday, level, position, salary, dno) VALUES (?, ?, ?, ?, ?, ?, ?)
2025-04-19 12:11:50,139 INFO sqlalchemy.engine.Engine [cached since 6.404s ago] ('0009', '测试', '1990-05-15', 3, '教师', 1000.0, '0001')
2025-04-19 12:11:50,141 INFO sqlalchemy.engine.Engine ROLLBACK
✗ 测试失败: (sqlite3.IntegrityError) CHECK constraint failed: salary BETWEEN 2000 AND 200000
[SQL: INSERT INTO emp (eno, ename, birthday, level, position, salary, dno) VALUES (?, ?, ?, ?, ?, ?, ?)]
[parameters: ('0009', '测试', '1990-05-15', 3, '教师', 1000.0, '0001')]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


In [18]:
def test_invalid_salary_high():
    """测试薪资上限约束"""
    emp = Emp(
        eno="0006", 
        ename="测试", 
        birthday=date(1990, 5, 15), 
        level=3, 
        position="教师", 
        salary=250000,  # 高于最高薪资
        dno="0001"
    )
    session.add(emp)
    session.commit()  # 应该引发异常
run_test("非法薪资值(超过上限)", test_invalid_salary_low)


测试: 非法薪资值(超过上限)
--------------------------------------------------
2025-04-19 12:11:51,167 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-19 12:11:51,167 INFO sqlalchemy.engine.Engine INSERT INTO emp (eno, ename, birthday, level, position, salary, dno) VALUES (?, ?, ?, ?, ?, ?, ?)
2025-04-19 12:11:51,168 INFO sqlalchemy.engine.Engine [cached since 7.433s ago] ('0009', '测试', '1990-05-15', 3, '教师', 1000.0, '0001')
2025-04-19 12:11:51,168 INFO sqlalchemy.engine.Engine ROLLBACK
✗ 测试失败: (sqlite3.IntegrityError) CHECK constraint failed: salary BETWEEN 2000 AND 200000
[SQL: INSERT INTO emp (eno, ename, birthday, level, position, salary, dno) VALUES (?, ?, ?, ?, ?, ?, ?)]
[parameters: ('0009', '测试', '1990-05-15', 3, '教师', 1000.0, '0001')]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


In [19]:
def test_invalid_dname():
    """测试部门名称枚举约束"""
    dept = Dept(
        dno="0009", 
        dname="物理学院",  # 不在枚举列表中
        budget=1000000, 
        manager="0001"
    )
    session.add(dept)
    session.commit()  # 应该引发异常

run_test("非法部门名称", test_invalid_dname)


测试: 非法部门名称
--------------------------------------------------
2025-04-19 12:11:52,124 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-19 12:11:52,124 INFO sqlalchemy.engine.Engine INSERT INTO dept (dno, dname, budget, manager) VALUES (?, ?, ?, ?)
2025-04-19 12:11:52,125 INFO sqlalchemy.engine.Engine [cached since 7.603s ago] ('0009', '物理学院', 1000000.0, '0001')
2025-04-19 12:11:52,126 INFO sqlalchemy.engine.Engine ROLLBACK
✗ 测试失败: (sqlite3.IntegrityError) CHECK constraint failed: dname IN ('数学学院', '计算机学院', '智能学院', '电子学院', '元培学院')
[SQL: INSERT INTO dept (dno, dname, budget, manager) VALUES (?, ?, ?, ?)]
[parameters: ('0009', '物理学院', 1000000.0, '0001')]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


In [20]:
def test_unique_key_emp():
    """测试员工主键唯一性约束"""
    # 插入已存在的员工编号
    emp = Emp(
        eno="0001",  # 已存在的ID
        ename="重复", 
        birthday=date(1995, 5, 15), 
        level=3, 
        position="教师", 
        salary=12000, 
        dno="0001"
    )
    session.add(emp)
    session.commit()  # 应该引发异常

run_test("重复员工主键", test_unique_key_emp)


测试: 重复员工主键
--------------------------------------------------
2025-04-19 12:11:53,144 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-19 12:11:53,144 INFO sqlalchemy.engine.Engine SELECT emp.eno AS emp_eno, emp.ename AS emp_ename, emp.birthday AS emp_birthday, emp.level AS emp_level, emp.position AS emp_position, emp.salary AS emp_salary, emp.dno AS emp_dno 
FROM emp 
WHERE emp.eno = ?
2025-04-19 12:11:53,146 INFO sqlalchemy.engine.Engine [generated in 0.00069s] ('0001',)
2025-04-19 12:11:53,148 INFO sqlalchemy.engine.Engine INSERT INTO emp (eno, ename, birthday, level, position, salary, dno) VALUES (?, ?, ?, ?, ?, ?, ?)
2025-04-19 12:11:53,149 INFO sqlalchemy.engine.Engine [cached since 9.413s ago] ('0001', '重复', '1995-05-15', 3, '教师', 12000.0, '0001')
2025-04-19 12:11:53,149 INFO sqlalchemy.engine.Engine ROLLBACK
✗ 测试失败: (sqlite3.IntegrityError) UNIQUE constraint failed: emp.eno
[SQL: INSERT INTO emp (eno, ename, birthday, level, position, salary, dno) VALUES (?, ?, ?, ?, ?, 

C:\Users\Dai\AppData\Local\Temp\ipykernel_14120\3584338704.py:14: SAWarning: New instance <Emp at 0x1c59d6b02e0> with identity key (<class '__main__.Emp'>, ('0001',), None) conflicts with persistent instance <Emp at 0x1c59d63e920>
  session.commit()  # 应该引发异常


In [21]:
def test_unique_key_dept():
    """测试部门主键唯一性约束"""
    # 插入已存在的部门编号
    dept = Dept(
        dno="0001",  # 已存在的ID
        dname="计算机学院", 
        budget=1000000, 
        manager="0001"
    )
    session.add(dept)
    session.commit()  # 应该引发异常

run_test("重复部门主键", test_unique_key_dept)


测试: 重复部门主键
--------------------------------------------------
2025-04-19 12:11:55,080 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-19 12:11:55,081 INFO sqlalchemy.engine.Engine SELECT dept.dno AS dept_dno, dept.dname AS dept_dname, dept.budget AS dept_budget, dept.manager AS dept_manager 
FROM dept 
WHERE dept.dno = ?
2025-04-19 12:11:55,082 INFO sqlalchemy.engine.Engine [generated in 0.00078s] ('0001',)
2025-04-19 12:11:55,084 INFO sqlalchemy.engine.Engine INSERT INTO dept (dno, dname, budget, manager) VALUES (?, ?, ?, ?)
2025-04-19 12:11:55,084 INFO sqlalchemy.engine.Engine [cached since 10.56s ago] ('0001', '计算机学院', 1000000.0, '0001')
2025-04-19 12:11:55,086 INFO sqlalchemy.engine.Engine ROLLBACK
✗ 测试失败: (sqlite3.IntegrityError) UNIQUE constraint failed: dept.dno
[SQL: INSERT INTO dept (dno, dname, budget, manager) VALUES (?, ?, ?, ?)]
[parameters: ('0001', '计算机学院', 1000000.0, '0001')]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


C:\Users\Dai\AppData\Local\Temp\ipykernel_14120\1215435416.py:11: SAWarning: New instance <Dept at 0x1c59d6b1990> with identity key (<class '__main__.Dept'>, ('0001',), None) conflicts with persistent instance <Dept at 0x1c59d6b3d30>
  session.commit()  # 应该引发异常


In [22]:
# 查询部门数据
for dept in session.query(Dept).all():
    print(dept)

2025-04-19 12:11:55,973 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-19 12:11:55,975 INFO sqlalchemy.engine.Engine SELECT dept.dno AS dept_dno, dept.dname AS dept_dname, dept.budget AS dept_budget, dept.manager AS dept_manager 
FROM dept
2025-04-19 12:11:55,976 INFO sqlalchemy.engine.Engine [generated in 0.00065s] ()
<Dept(dno='0001', dname='计算机学院', budget=1000000.0, manager='0001')>
<Dept(dno='0002', dname='数学学院', budget=800000.0, manager='0002')>
<Dept(dno='0003', dname='智能学院', budget=1200000.0, manager='0003')>
<Dept(dno='0005', dname='电子学院', budget=1200000.0, manager='0004')>
<Dept(dno='0004', dname='元培学院', budget=900000.0, manager='0007')>


In [23]:
# 查询员工数据
for emp in session.query(Emp).all():
    print(emp)

# 清理资源
session.close()
print("会话已关闭")

2025-04-19 12:11:56,820 INFO sqlalchemy.engine.Engine SELECT emp.eno AS emp_eno, emp.ename AS emp_ename, emp.birthday AS emp_birthday, emp.level AS emp_level, emp.position AS emp_position, emp.salary AS emp_salary, emp.dno AS emp_dno 
FROM emp
2025-04-19 12:11:56,821 INFO sqlalchemy.engine.Engine [generated in 0.00086s] ()
<Emp(eno='0001', ename='张三', level=4, position='教师', salary=15000.0, dno='0001')>
<Emp(eno='0002', ename='李四', level=3, position='教务', salary=8000.0, dno='0002')>
<Emp(eno='0003', ename='王五', level=5, position='教师', salary=20000.0, dno='0003')>
<Emp(eno='0004', ename='吴十', level=3, position='秘书', salary=7000.0, dno='0003')>
<Emp(eno='0006', ename='吴六', level=2, position='教师', salary=12000.0, dno='0001')>
<Emp(eno='0005', ename='赵六', level=3, position='秘书', salary=7000.0, dno='0001')>
<Emp(eno='0007', ename='钱七', level=5, position='教师', salary=25000.0, dno='0004')>
2025-04-19 12:11:56,822 INFO sqlalchemy.engine.Engine ROLLBACK
会话已关闭
